In [ ]:
# pip install boto3

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.functions import monotonically_increasing_id

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("dim_payment_format") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.profile",
        "default"
    ) \
    .config("spark.driver.memory", "10g") \
    .config("spark.driver.memoryOverhead", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

In [3]:
spark.sparkContext._jsc.hadoopConfiguration().set(
    "fs.s3a.aws.credentials.provider",
    "com.amazonaws.auth.profile.ProfileCredentialsProvider"
)

spark.sparkContext._jsc.hadoopConfiguration().set(
    "fs.s3a.aws.profile",
    "default"
)

In [4]:
transactions = spark.read.parquet(
    "s3a://datapath-buckets/bank_datasets/curated/transactions/"
)

accounts = spark.read.parquet(
    "s3a://datapath-buckets/bank_datasets/curated/accounts/"
)

In [5]:
transactions.show(5)
accounts.show(5)

+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+
|          Timestamp|From Bank|Account_From|To Bank|Account_To|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|minute_part|day_of_month|year|month|
+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+-----------+------------+----+-----+
|2022-09-01 00:16:00|        1|   8000EC1E0|      1| 8000EC1E0|          11.86|         US Dollar|      11.86|       US Dollar|  Reinvestment|            0|         16|           1|2022|    9|
|2022-09-01 00:04:00|        1|   8000F4510|  11813| 8011305D0|           9.82|         US Dollar|       9.82|       US Dollar|   Credit Card|            0|          4|           1|2022|    9|
|2022-09-01 00:11:00|       12|   8

In [6]:
transactions.printSchema()
accounts.printSchema()

root
 |-- Timestamp: timestamp (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account_From: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account_To: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)
 |-- minute_part: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

root
 |-- Bank Name: string (nullable = true)
 |-- Bank ID: integer (nullable = true)
 |-- Account Number: string (nullable = true)
 |-- Entity ID: string (nullable = true)
 |-- Entity Name: string (nullable = true)



In [11]:


payment = transactions.select( col("Payment Format").alias("Payment")).distinct().withColumn("Payment_ID", monotonically_increasing_id())
payment.show(20)


+------------+----------+
|     Payment|Payment_ID|
+------------+----------+
| Credit Card|         0|
|         ACH|         1|
|        Cash|         2|
|        Wire|         3|
|     Bitcoin|         4|
|Reinvestment|         5|
|      Cheque|         6|
+------------+----------+



In [12]:
payment.write \
    .mode("overwrite") \
    .parquet("s3a://datapath-buckets/bank_datasets/curated/dimensional_model/dim_payment_format/")